# A vignette: using the trained Electome Factor models

This notebook is a short, runnable walkthrough of how to use the six paper-active Electome Factor (EF) models shipped in `models/`. It is designed to be self-contained — you do **not** need any of the lab data files on RDSS to run it. A tiny LFP-shaped simulated dataset is generated by `examples/simulated_data.py` so every cell below executes in seconds.

What you'll do here:

1. **Pick an EF** out of the six and load it with one call.
2. **Project** simulated LFP-spectral windows through the encoder to obtain loading scores `s[:, 0]`.
3. **Compute per-mouse AUC** of those scores against a binary label
   (the simulated `onnest_label`).
4. **Inspect the feature space** with a scree plot of the cumulative
   squared-L2 contribution of factor 0.
5. **Visualize the selected features** with the dual-filter bar (3-band)
   or dot (1-Hz) heatmap — automatically chosen based on the model's
   frequency resolution.

For the full training pipeline (Stage-1 LOO hyperparameter selection,
Stage-2 pure LOO AUC reporting, and Stage-3 paper-model training) see the
six task notebooks in `notebooks/` — each follows the same 8-section
structure described in the top-level `README.md`.

In [ ]:
# Allow imports from ../src and ../examples
import sys, os
_here = os.path.abspath('')
_repo = os.path.abspath(os.path.join(_here, os.pardir))
for p in (os.path.join(_repo, 'src'), os.path.join(_repo, 'examples')):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import matplotlib.pyplot as plt


## Section 2. The six EF models

Each of the six task notebooks in `notebooks/` produces one frozen
`.pt` model, all of which live in `models/`. The table below summarizes
what each one is for; see `models/README.md` for full per-model
hyperparameter details and the old-vs-new filename mapping.

| Filename in `models/` | EF | Task | Frequency resolution |
| --- | --- | --- | --- |
| `OnnestVsOffnest_3band.pt` | Maternal Engagement | On-nest vs off-nest LFP windows | 3 wide bands |
| `OnnestVsOffnest_1Hz.pt` | Maternal Engagement | On-nest vs off-nest LFP windows | 54 × 1-Hz bins |
| `LickingVsNonLicking_3band.pt` | Licking | Licking vs non-licking (within on-nest) | 3 wide bands |
| `LickingVsGrooming_3band.pt` | Licking-vs-Grooming | Licking vs self-grooming (within on-nest) | 3 wide bands |
| `PreVsPost134_3band.pt` | Maternal Stage | Pre home vs P1/P3/P4 home | 3 wide bands |
| `PreVsPost134_1Hz.pt` | Maternal Stage | Pre home vs P1/P3/P4 home | 54 × 1-Hz bins |

In [ ]:
from models_registry import list_ef_models, get_model_info

for name in list_ef_models():
    info = get_model_info(name)
    print(f'  {name:30s}  {info["ef_name"]}')


## Section 3. Load an EF model

Loading any of the six is a single call to `load_ef_model(name)`. The
function locates `models/` for you (regardless of whether you launch
Jupyter from `examples/`, from the repo root, or from `notebooks/`) and
unpickles the model object directly onto CPU.

For the rest of this walkthrough we'll use the 3-band Maternal-Engagement
EF — it's the smallest model and the easiest to interpret. At the very
end of the notebook we'll also load the 1-Hz version to show how the
element-selection heatmap switches from bars to dots.

In [ ]:
from models_registry import load_ef_model

model = load_ef_model('OnnestVsOffnest_3band')
print(f'h = {model.h}')
print(f'dim_in = {model.dim_in}')
print(f'n_components = {model.n_components}')
print(f'sup_weight (μ) = {model.sup_weight}')


## Section 4. Load the simulated demo dataset

The dataset is intentionally tiny — **3 fake mice × 50 windows each =
150 windows** spread across five maternal stages (`Pre`, `P1`, `P3`,
`P8`, `P14`). Features follow LFP-shaped distributions (lognormal
per-band power, beta-distributed squared coherence) so the loaded model
actually produces meaningful scores instead of NaNs.

Only windows whose `period` is in `{P1, P3, P8}` carry an `onnest_label`
(`np.nan` elsewhere) — the on-nest classification task is only meaningful
at active maternal-care stages, so labels are only present there. The
simulator injects a small theta-band power and coherence bias for
`onnest_label == 1` windows (and flips the label assignment if the
model's score sign would otherwise come out backwards), so that the
per-mouse AUC has a clear separation rather than sitting at chance.

The full generator lives in `examples/simulated_data.py` — read it to
see exactly what's being simulated.

In [ ]:
from simulated_data import load_demo_data

data = load_demo_data(model)

print(f'X shape       : {data["X"].shape}  (expected (N, dim_in) = (150, {model.dim_in}))')
print(f'mice          : {sorted(set(data["mouse_id"]))}')
print(f'stages        : {sorted(set(data["period"]))}')
print(f'with onnest y : {int((~np.isnan(data["onnest_label"])).sum())}/{len(data["mouse_id"])} windows')


## Section 5. Backprojection and per-mouse AUC

The trained encoder is a frozen, deterministic projection from the
feature space (108 columns for the 3-band model) to the 10-dimensional
latent factor space. The first latent dimension is the supervised one
— that's `s[:, 0]` — and it is the loading score you'd plot in the
paper figures.

`compute_loading_scores(model, X)` is the one-line wrapper around the
model's `predict_proba(..., include_scores=True)` call. It puts the
model in eval mode, runs a no-grad forward pass, and returns just
`s[:, 0]` as a 1-D numpy array.

`compute_per_mouse_auc(scores, y_true, mouse_ids)` computes the ROC-AUC
per mouse, skipping any mouse that ends up single-class after dropping
NaN labels. Note: on simulated data the AUC will be ~0.6-1.0
(depending on which EF you loaded) — on real recordings the same code
would produce the paper-reported numbers.

In [ ]:
from workflow import compute_loading_scores, compute_per_mouse_auc

scores = compute_loading_scores(model, data['X'])
aucs   = compute_per_mouse_auc(scores, data['onnest_label'], data['mouse_id'])

print(f'scores: shape={scores.shape}, range=[{scores.min():.3f}, {scores.max():.3f}]')
print('per-mouse AUC:')
for mid, auc in aucs.items():
    print(f'  {mid}: {auc:.3f}')
print(f'mean: {np.mean(list(aucs.values())):.3f}')


### Visualize the scores

Three plots from `src/viz.py` are useful for inspecting how a frozen EF
model behaves on new recordings:

* `plot_per_mouse_timeseries` — the chronological time series for a
  single mouse, color-coded by period. Useful for spotting whether the
  loading score tracks any per-window structure within a recording.
* `plot_per_stage_boxplot` — the score distribution per stage pooled
  across mice. The on-nest model is expected to give higher scores in
  P1/P3/P8 than in Pre.
* `plot_per_mouse_auc_bar` — horizontal bars of the per-mouse AUC; a
  reference line marks chance.


In [ ]:
from viz import (plot_per_mouse_timeseries,
                  plot_per_stage_boxplot,
                  plot_per_mouse_auc_bar)

plot_per_mouse_timeseries(scores, data['period'], data['mouse_id'],
                          mouse_id_to_show='SimMouse_01')
plt.show()

plot_per_stage_boxplot(scores, data['period'],
                       stages_order=['Pre', 'P1', 'P3', 'P8', 'P14'])
plt.show()

plot_per_mouse_auc_bar(aucs)
plt.show()


## Section 6. Element selection — scree plot

Once you've picked an EF, the next paper-figure question is *which*
input features actually drive factor 0 (the supervised latent). The
scree plot below sorts every feature in `W[0, :]` by descending
squared-L2 contribution and shows the running cumulative fraction —
the curve at the bottom-right tells you how many features you need to
cover any particular fraction of the total squared-L2 energy.

The horizontal dotted line marks `threshold_ratio = 0.9`, which is the
default cutoff used by `process_W_nmf_k` and by
`process_W_nmf_dual_filter` to decide which features are "important".
The vertical dashed line tells you how many features that cutoff
corresponds to for this particular model — typically a small minority
of the 108 (3-band) or 1944 (1-Hz) input features.

In [ ]:
from viz import plot_scree_W_nmf

plot_scree_W_nmf(model.get_W_nmf(), k=0, threshold_ratio=0.9)
plt.show()


## Section 7. Element selection — dual-filter heatmap

Beyond the scree plot, the paper figures use a dual-filter view that
combines:

* **Absolute strength** — features whose cumulative squared-L2
  contribution to factor 0 is in the top 90 % (dark = strong).
* **Relative uniqueness** — for each feature, the fraction of the
  inter-factor sum that is attributable to factor 0 (≥ 0.5 means
  factor 0 is the dominant explainer of that feature among the 10
  components).

`plot_dual_filter(model, train_dict)` runs both filters via
`process_W_nmf_dual_filter` and then chooses the appropriate heatmap
style automatically:

* **3-band model (3 frequency columns)** → **bar heatmap** —
  bar length encodes absolute strength, bar color encodes relative
  uniqueness.
* **1-Hz model (54 frequency columns)** → **dot heatmap** —
  bars would be too narrow to read with 54 columns, so dot size encodes
  absolute strength and dot color encodes relative uniqueness instead.

Both styles use the same Iowa-Gold border to mark features that pass
*both* filters simultaneously.

### 7a. 3-band → bar heatmap

In [ ]:
from viz import plot_dual_filter

fig, *_ = plot_dual_filter(model, data)
plt.show()


### 7b. 1-Hz → dot heatmap

Now load the 1-Hz Maternal-Engagement EF (same task, but with 54 × 1-Hz
frequency bins instead of 3 wide bands) and re-run the same one-line
wrapper — `plot_dual_filter` automatically switches to the dot heatmap
because the number of frequency columns is > 10.

In [ ]:
model_1hz = load_ef_model('OnnestVsOffnest_1Hz')
data_1hz  = load_demo_data(model_1hz)

fig, *_ = plot_dual_filter(model_1hz, data_1hz)
plt.show()
